In [1]:
import os
import sys

# Colab: clone the repo once, then work inside it so `data/` is the GitHub project folder:
#   !git clone https://github.com/gavealfan/cs288-final-project.git
#   %cd cs288-final-project
# (Optional) upload this notebook into that folder, or open the notebook from GitHub in Colab.

# Prevent transformers from importing TensorFlow/Keras on local setups.
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'

# Install all training dependencies (run this once per fresh runtime)
# Pin transformers<5 (API drift). Pin trl>=0.12,<0.13 so SFTTrainer still accepts max_seq_length= (T4 VRAM); 0.13+ moved it to SFTConfig only.
!{sys.executable} -m pip install -q "transformers>=4.40,<5" "datasets>=2.18" "trl>=0.12,<0.13" "peft>=0.10" "accelerate>=0.28" "bitsandbytes>=0.43" "wandb" "sentencepiece" "tiktoken"

print("Dependencies installed. If imports still fail, restart runtime/kernel and rerun from top.")
print("Colab: Runtime -> Change runtime type -> GPU (T4). Qwen2.5-7B + 4-bit LoRA fits ~16GB with seq length 2048.")
print("If download fails: accept the Qwen license on huggingface.co, then run: !huggingface-cli login")

Dependencies installed. If imports still fail, restart runtime/kernel and rerun from top.


In [2]:
# Step 2: Config + data files (teacher SFT JSONL from the pipeline, or legacy DPO pairs)
import os
from pathlib import Path


def _repo_root() -> Path:
    """Resolve the cloned repo root so all paths use the GitHub project folder."""
    if os.environ.get('REPO_ROOT'):
        return Path(os.environ['REPO_ROOT']).expanduser().resolve()
    cwd = Path.cwd().resolve()
    if (cwd / '.git').is_dir() or (cwd / 'data').is_dir():
        return cwd
    for p in (Path('/content/cs288-final-project'), Path('/content/gavealfan/cs288-final-project')):
        rp = p.resolve()
        if (rp / '.git').is_dir() or (rp / 'data').is_dir():
            return rp
    return cwd


def _normalize_upload_name(name: str, data_source: str) -> str:
    """Map Colab-uploaded names like 'file (1).jsonl' to canonical expected filenames."""
    lower = name.lower()
    if data_source == 'teacher_sft':
        if 'teacher_sft_train' in lower:
            return 'teacher_sft_train.jsonl'
        if 'teacher_sft_eval' in lower:
            return 'teacher_sft_eval.jsonl'
    else:
        if 'dpo_pairs_train' in lower:
            return 'dpo_pairs_train.jsonl'
        if 'dpo_pairs_eval' in lower:
            return 'dpo_pairs_eval.jsonl'
    return Path(name).name


REPO_ROOT = _repo_root()
os.chdir(REPO_ROOT)
DATA_DIR = REPO_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
data_dir = str(DATA_DIR)

# Colab: train the real student (Qwen2.5-7B). Set LOCAL_DEBUG=1 in env only for a tiny smoke test.
LOCAL_DEBUG = os.environ.get('LOCAL_DEBUG', '0') == '1'

# T4 (16GB): 2048 + 7B QLoRA + batch 1 + grad checkpointing is usually safe; set 1024 if OOM.
MAX_SEQ_LENGTH = int(os.environ.get('MAX_SEQ_LENGTH', '2048'))

# "teacher_sft" -> teacher_sft_*.jsonl | "dpo_pairs" -> dpo_pairs_*.jsonl then convert to distill_*.jsonl
DATA_SOURCE = os.environ.get('DATA_SOURCE', 'teacher_sft').strip().lower()
if DATA_SOURCE not in {'teacher_sft', 'dpo_pairs'}:
    raise ValueError("DATA_SOURCE must be one of: 'teacher_sft', 'dpo_pairs'.")

if DATA_SOURCE == 'teacher_sft':
    train_path = DATA_DIR / 'teacher_sft_train.jsonl'
    eval_path = DATA_DIR / 'teacher_sft_eval.jsonl'
    need_names = 'teacher_sft_train.jsonl and teacher_sft_eval.jsonl'
else:
    train_path = DATA_DIR / 'dpo_pairs_train.jsonl'
    eval_path = DATA_DIR / 'dpo_pairs_eval.jsonl'
    need_names = 'dpo_pairs_train.jsonl and dpo_pairs_eval.jsonl'

if DATA_SOURCE == 'teacher_sft':
    TRAIN_JSONL = str(train_path)
    EVAL_JSONL = str(eval_path)
else:
    TRAIN_JSONL = str(DATA_DIR / 'distill_train.jsonl')
    EVAL_JSONL = str(DATA_DIR / 'distill_eval.jsonl')

missing = [p for p in (train_path, eval_path) if not p.exists()]
if not missing:
    print('Found data files:')
    print(f'  {train_path}')
    print(f'  {eval_path}')
    print(f'REPO_ROOT={REPO_ROOT} | data_dir={data_dir}')
    print(
        f'DATA_SOURCE={DATA_SOURCE} | LOCAL_DEBUG={LOCAL_DEBUG} | MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} | TRAIN_JSONL={TRAIN_JSONL}'
    )
else:
    try:
        from google.colab import files

        print(f'Upload {need_names} (or place them under {DATA_DIR} and rerun).')

        while True:
            missing = [p for p in (train_path, eval_path) if not p.exists()]
            if not missing:
                break

            print('Still missing:')
            for p in missing:
                print(f'  - {p.name}')

            uploaded = files.upload()
            if not uploaded:
                print('No files selected. Please upload the missing files listed above.')
                continue

            for name, content in uploaded.items():
                normalized = _normalize_upload_name(name, DATA_SOURCE)
                dest = DATA_DIR / normalized
                dest.write_bytes(content)
                print(f'  Saved {dest}')

        print('Found data files:')
        print(f'  {train_path}')
        print(f'  {eval_path}')
        print(f'REPO_ROOT={REPO_ROOT} | data_dir={data_dir}')
        print(
            f'DATA_SOURCE={DATA_SOURCE} | LOCAL_DEBUG={LOCAL_DEBUG} | MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} | TRAIN_JSONL={TRAIN_JSONL}'
        )
    except ModuleNotFoundError:
        raise RuntimeError(
            f'Files missing: {missing}. Put them in {DATA_DIR}, or set REPO_ROOT to your clone path, then rerun.'
        ) from None

Found existing data files, skipping upload step:
  data/dpo_pairs_train.jsonl
  data/dpo_pairs_eval.jsonl


In [3]:
# Step 3: Verify data
import json


def count_lines(path):
    with open(path) as f:
        return sum(1 for _ in f)


train_count = count_lines(TRAIN_JSONL)
eval_count = count_lines(EVAL_JSONL)
print(f'Training examples: {train_count}')
print(f'Eval examples: {eval_count}')
if train_count == 0 or eval_count == 0:
    raise RuntimeError('Train/eval jsonl is empty. Rebuild or re-upload files before training.')

with open(TRAIN_JSONL) as f:
    example = json.loads(f.readline())
print(f'\nExample prompt (first 200 chars):\n{example["prompt"][:200]}')
if 'response' in example:
    print(f'\nResponse (first 200 chars):\n{example["response"][:200]}')
if 'chosen' in example:
    print(f'\nChosen (first 150 chars):\n{example["chosen"][:150]}')
    print(f'\nRejected (first 150 chars):\n{example["rejected"][:150]}')

Training pairs: 1054
Eval pairs: 262

Example prompt (first 200 chars):
You are in the following organizational role:
You are a team lead in human resources, who reports to the person they are speaking with. You recently joined the company (less than 6 months).

Interacti

Chosen (first 150 chars):
Hi [Manager's Name], I wanted to give you a quick update on Project Elevate. Mark from IT didn't make it to our scheduled platform demo yesterday, and

Rejected (first 150 chars):
Hey, I need to talk to you about Mark. He completely blew off our demo call yesterday without a word. This is unacceptable, especially considering how


In [4]:
import json

# Only needed when DATA_SOURCE == "dpo_pairs" (teacher response = chosen).
if DATA_SOURCE != 'dpo_pairs':
    print('Skipping conversion (DATA_SOURCE=teacher_sft already has prompt+response).')
else:
    INPUT_TRAIN = str(DATA_DIR / 'dpo_pairs_train.jsonl')
    INPUT_EVAL = str(DATA_DIR / 'dpo_pairs_eval.jsonl')
    OUTPUT_TRAIN = str(DATA_DIR / 'distill_train.jsonl')
    OUTPUT_EVAL = str(DATA_DIR / 'distill_eval.jsonl')

    def convert_to_distill(input_path, output_path):
        with open(input_path) as fin, open(output_path, 'w') as fout:
            for line in fin:
                data = json.loads(line)
                new_data = {'prompt': data['prompt'], 'response': data['chosen']}
                fout.write(json.dumps(new_data) + '\n')
        print(f'Saved {output_path}')

    convert_to_distill(INPUT_TRAIN, OUTPUT_TRAIN)
    convert_to_distill(INPUT_EVAL, OUTPUT_EVAL)

Saved data/distill_train.jsonl
Saved data/distill_eval.jsonl


In [5]:
# Step 4: Load model and tokenizer (Colab: Qwen2.5-7B + 4-bit QLoRA; LOCAL_DEBUG -> distilgpt2)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

if LOCAL_DEBUG:
    MODEL_NAME = 'distilgpt2'
else:
    # Same base as train_sft_local.py / experiment student_profile qwen7b
    MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

print(f'Loading {MODEL_NAME}...')
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
except ValueError:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

has_cuda = torch.cuda.is_available()
USE_BF16 = False
USE_FP16 = has_cuda

use_4bit = has_cuda and not LOCAL_DEBUG
bnb_config = None
if use_4bit:
    # T4-friendly: NF4 + FP16 compute + double quant saves VRAM vs plain 4-bit.
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

torch_dtype = torch.float16 if has_cuda else torch.float32
kwargs = {
    'torch_dtype': torch_dtype,
    'device_map': 'auto',
}
if bnb_config is not None:
    kwargs['quantization_config'] = bnb_config

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kwargs)
print('Model loaded.')
if has_cuda:
    props = torch.cuda.get_device_properties(0)
    print(
        f'GPU: {props.name} | {props.total_memory / 1e9:.1f} GB total — T4 works with this stack (7B QLoRA, seq cap {MAX_SEQ_LENGTH}).'
    )
    print(f'USE_BF16={USE_BF16} | USE_FP16={USE_FP16} | use_4bit={use_4bit}')
    print(f'GPU memory used after load: {torch.cuda.memory_allocated() / 1e9:.1f} GB')
else:
    print('No CUDA: running on CPU (slow). Enable GPU in Colab: Runtime -> Change runtime type -> T4.')

/Users/tobyli/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading distilgpt2...


`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded.
Running on CPU.


In [6]:
from datasets import Dataset

def load_sft_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            if not line.strip():
                continue
            data = json.loads(line)
            records.append({'prompt': data['prompt'], 'response': data['response']})
    return Dataset.from_list(records)


train_dataset = load_sft_jsonl(TRAIN_JSONL)
eval_dataset = load_sft_jsonl(EVAL_JSONL)


def format_example(example):
    return {'text': example['prompt'] + '\n\n' + example['response']}


train_dataset = train_dataset.map(format_example)
eval_dataset = eval_dataset.map(format_example)

if LOCAL_DEBUG:
    train_dataset = train_dataset.select(range(min(32, len(train_dataset))))
    eval_dataset = eval_dataset.select(range(min(8, len(eval_dataset))))

print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)} | MODEL_NAME={MODEL_NAME}')

Map: 100%|██████████| 262/262 [00:00<00:00, 55663.44 examples/s]

Train: 32 | Eval: 8


In [7]:
from peft import LoraConfig
from transformers import TrainingArguments
from trl import SFTTrainer

if LOCAL_DEBUG:
    peft_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=['c_attn', 'c_proj'],
        task_type='CAUSAL_LM',
    )
else:
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        task_type='CAUSAL_LM',
    )

training_args = TrainingArguments(
    output_dir='checkpoints/role-conditioned-distill',
    learning_rate=2e-5 if not LOCAL_DEBUG else 5e-6,
    num_train_epochs=1 if LOCAL_DEBUG else 2,
    max_steps=10 if LOCAL_DEBUG else -1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1 if LOCAL_DEBUG else 8,
    logging_steps=1 if LOCAL_DEBUG else 10,
    eval_strategy='steps',
    eval_steps=5 if LOCAL_DEBUG else 50,
    save_steps=10 if LOCAL_DEBUG else 200,
    save_total_limit=2,
    fp16=USE_FP16,
    bf16=False,
    report_to='none',
    gradient_checkpointing=not LOCAL_DEBUG,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    optim='adamw_torch',
)

if training_args.gradient_checkpointing:
    model.enable_input_require_grads()

print('Config ready.')
print(f'  LOCAL_DEBUG={LOCAL_DEBUG} | MODEL_NAME={MODEL_NAME} | DATA_SOURCE={DATA_SOURCE}')
print(f'  MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} | USE_BF16={USE_BF16} | USE_FP16={USE_FP16}')
print(
    f'  Effective train batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}'
)

Config ready.
  LOCAL_DEBUG: True
  Effective batch size: 1


In [ ]:
# Optional: avoid PEFT "reload adapter" warnings if you rerun this cell
try:
    if hasattr(model, 'unload'):
        model.unload()
except Exception:
    pass

train_dataset = train_dataset.remove_columns(
    [c for c in train_dataset.column_names if c != 'text']
)
eval_dataset = eval_dataset.remove_columns(
    [c for c in eval_dataset.column_names if c != 'text']
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    processing_class=tokenizer,
    peft_config=peft_config,
    formatting_func=lambda x: x['text'],
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

print('Starting SFT (teacher distillation) training...')
trainer.train()
print('Training complete!')

/Users/tobyli/miniconda3/lib/python3.13/site-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
Tokenizing eval dataset: 100%|██████████| 8/8 [00:00<00:00, 1793.68 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
/Users/tobyli/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Starting self distillation training...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
5,3.919800,3.609785
10,3.906300,3.609685


Training complete!


In [9]:
# Step 8: Save model
OUTPUT_DIR = str(REPO_ROOT / 'checkpoints' / 'role-conditioned-distill-final')
ZIP_NAME = str(REPO_ROOT / 'role_conditioned_self_distillation_lora.zip')

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

# Create downloadable archive
!zip -r "{ZIP_NAME}" "{OUTPUT_DIR}"

# Download only if running in Colab
try:
    from google.colab import files
    files.download(ZIP_NAME)
except ModuleNotFoundError:
    print(f'Not in Colab. Saved archive locally as {ZIP_NAME}.')

Model saved to checkpoints/role-conditioned-distill-final
  adding: checkpoints/role-conditioned-distill-final/ (stored 0%)
  adding: checkpoints/role-conditioned-distill-final/adapter_model.safetensors (deflated 7%)
  adding: checkpoints/role-conditioned-distill-final/tokenizer_config.json (deflated 54%)
  adding: checkpoints/role-conditioned-distill-final/special_tokens_map.json (deflated 60%)
  adding: checkpoints/role-conditioned-distill-final/tokenizer.json (deflated 82%)
  adding: checkpoints/role-conditioned-distill-final/README.md (deflated 65%)
  adding: checkpoints/role-conditioned-distill-final/merges.txt (deflated 53%)
  adding: checkpoints/role-conditioned-distill-final/training_args.bin (deflated 53%)
  adding: checkpoints/role-conditioned-distill-final/adapter_config.json (deflated 59%)
  adding: checkpoints/role-conditioned-distill-final/vocab.json (deflated 59%)
Not in Colab. Saved archive locally as role_conditioned_self_distillation_lora.zip.


In [10]:
# Step 9: Quick inference test
from peft import PeftModel

test_prompt = """You are a middle manager in engineering, who outranks the person they are speaking with. You have 1-5 years at the company.

Interaction context:
You are assigning a task to someone. Describe the task clearly, set expectations, and provide any necessary context.

Situation: The team needs to migrate the authentication service from the legacy monolith to a microservice architecture before the Q3 deadline.
Background: The migration has been planned for months but keeps getting deprioritized. The engineer you're assigning this to is capable but has a full plate.

Respond appropriately given your role and the situation."""

_device = next(model.parameters()).device
inputs = tokenizer(test_prompt, return_tensors='pt').to(_device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
    )
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('Generated response:')
print(response)

Generated response:

Situation: the company needs to migrate to the legacy monolith.
Context: In a language like Java, the team has to migrate the authentication service from the legacy monolith to a microservice architecture before the Q3 deadline.
Context: The team needs to migrate to the legacy monolith to a microservice architecture before the Q3 deadline.
Expect a lot of attention from the team.
Respond appropriately given your role and the situation.
Situation: The company needs to migrate to the legacy monolith to a microservice architecture before the Q3 deadline.
Context: The team needs to migrate to the legacy monolith to a microservice architecture before the Q3 deadline.
Context: The team needs to migrate to the legacy monolith to a microservice architecture before the Q3 deadline.
Expect a lot of attention from the team.
The team needs to migrate to the legacy monolith to a microservice architecture before the Q3 deadline.
Expect a lot of attention from the team.
Respond a